# ReFuelEU optimisation — migration validation

Port of the `optim_backwards` publication notebooks onto `main`'s generic energy
carriers and markets models. Three notebooks:

| | |
|---|---|
| **`00_migration_validation.ipynb`** | this one — does the port reproduce the paper? |
| `01_optimisation_runs.ipynb` | the five cases × ten carbon budgets |
| `02_results.ipynb` | the paper's figures |

Shared scenario construction lives in `optimisation_runs.py`, the six constraints in
`constraints_rte.py`.

**What changed, and what did not.** The published problem drives two design variables,
the biofuel and electrofuel blending shares. Underneath, the old model carried five
biofuel pathways at *constant* sub-shares, so the five collapse into one generic
pathway with no loss of fidelity:

| pathway | share | MFSP (€/L) |
|---|---|---|
| hefa_fog | 0.6 % | 0.815488 |
| hefa_others | 12.5 % | 1.052703 |
| ft_msw | 6.6 % | 1.142423 |
| ft_others | 68.9 % | 1.378082 |
| atj | 11.4 % | 1.38668 |

Blend-weighted mean = **1.3195 €/L**, against the 1.31 €/L reported in §3.1.

The electrofuel chain was rederived the same way: electrolysis 0.59 × H₂→fuel 0.74
gives 2.2904 MJ_elec/MJ_fuel, so the grid factor of 205 → 12 gCO₂/kWh maps to
**130.4 → 7.6 gCO₂/MJ**, matching the "130 to 7" of §3.1.

## 0. Setup

Run from this directory: the config resolves its relative paths against itself.

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import gemseo as gm
from aeromaps.utils.functions import custom_logger_config

import optimisation_runs as R

warnings.filterwarnings("ignore")

# Same call as the published notebooks. GEMSEO's default level is INFO (20), which is
# what prints the optimisation iteration log - lower it to WARNING and the run goes
# silent. custom_logger_config only patches NumPy-docstring parsing; it suppresses
# nothing.
custom_logger_config(gm.configure_logger())

## 1. One MDA at the ReFuelEU-linear design point

The paper publishes values for this design point (Table 3, and the −10.4 Bn€ label on
Figure 9), so it is the natural target. `config_rte.yaml` keeps the paper's four
markets.

In [ ]:
process = R.build_process("main")
R.set_mandate(process, **R.REFUELEU_MANDATE)
process.compute()

print(f"{len(process.mda_chain.disciplines)} disciplines")

In [ ]:
vector = process.data["vector_outputs"]

# Values produced by the same design point on branch optim_backwards.
reference = {
    "Airfare 2050 (EUR/RPK)": (vector["airfare_per_rpk"].loc[2050], 0.10080444, "0.101 (Table 3)"),
    "RPK 2050": (vector["rpk"].loc[2050], 2.2159493e12, "~2.2e12 (Figure 8)"),
    "Cumulative CO2 (Gt)": (
        vector["cumulative_co2_emissions"].loc[2050],
        3.8650519,
        "3.87 (Table 3)",
    ),
    "Surplus loss (Bn EUR)": (
        vector["cumulative_total_surplus_loss_discounted"].loc[2050] / 1e9,
        -10.388626,
        "-10.4 (Figure 9)",
    ),
}

rows = [
    [name, f"{new:.8g}", f"{old:.8g}", f"{(new - old) / abs(old) * 100:+.4f} %", paper]
    for name, (new, old, paper) in reference.items()
]
print(
    pd.DataFrame(
        rows, columns=["indicator", "migrated", "optim_backwards", "delta", "paper"]
    ).to_string(index=False)
)

### Reading the deltas

Airfare, RPK and energy agree to about 1e-6 — MDA tolerance, i.e. as close as the two
branches can get. Cumulative CO₂ is out by −0.09 % for one known and deliberate reason,
below.

Getting there took four corrected constants. The as-migrated port looked accurate
(+0.26 % on surplus loss) but was not: two of its errors happened to cancel. Applied
one at a time, against `optim_backwards`:

| | as migrated | +kerosene EF | +biofuel EF | +airfare anchor | +LHV 35.3 |
|---|---|---|---|---|---|
| airfare 2050 | +0.055 % | +0.055 % | +0.055 % | +0.044 % | +0.0001 % |
| RPK 2050 | +0.097 % | +0.097 % | +0.097 % | −0.039 % | −0.0001 % |
| cumulative CO₂ | +0.310 % | −0.005 % | +0.001 % | −0.096 % | −0.0000 % |
| surplus loss | +0.257 % | +0.402 % | +0.406 % | **+2.332 %** | +0.015 % |

Surplus loss is the sensitive one because it is a small residual of two large,
nearly-cancelling quantities — it amplifies an input error roughly 25×. It is the
indicator to watch when changing anything in this scenario.

The four constants, all now in the data files:

* **fossil kerosene emission factor** 88.7, not 89.0 gCO₂/MJ — `energy_rte.yaml`.
* **biofuel emission factor** 20.8269, not 20.8 gCO₂/MJ — same file.
* **MFSP conversion basis 35.3 MJ/L**, not 0.8 × 44 = 35.2. The legacy cost models
  hardcoded 35.3 while the *mass* side used `lhv_kerosene = 44` with density 0.8;
  legacy was internally inconsistent and only its price side used 35.3. So `lhv` stays
  44 and only the €/MJ values were rescaled.
* **`initial_airfare_per_rpk` = 0.09236379319842411** — `markets_rte.yaml`. This
  anchors both the demand elasticity and the inverse supply function, which must agree
  on one price. The other constant floating around the legacy code,
  0.09251431471704129, is a superseded calibration variant (a $ markup applied to a €
  cost, undeflated); it was inert there because it was only ever a solver seed.

**The remaining −0.09 % on cumulative CO₂ is the pre-2025 mandate ramp**, left as is.
`main` interpolates the biofuel mandate linearly 0 → 2 % over 2020–2025; legacy held it
at 0 through 2023 then jumped to 0.5 % in 2024. From 2025 on the two are identical. It
is a convention difference worth ~0.0035 Gt and touches no cost.

## 2. The collapsed single-market variant

`config_1m.yaml` merges the three passenger markets into one: 96 disciplines instead of
106, results identical to 7 significant figures. The split is inert here because the
paper gives all three passenger markets the same growth and the same efficiency
assumptions — worth knowing if you want a cheaper chain to experiment on.

In [ ]:
process_1m = R.build_process("main", config="config_1m.yaml")
R.set_mandate(process_1m, **R.REFUELEU_MANDATE)
process_1m.compute()

print(f"{len(process_1m.mda_chain.disciplines)} disciplines")
print(f"airfare 2050  4 markets {process.data['vector_outputs']['airfare_per_rpk'].loc[2050]:.8f}")
print(
    f"              1 market  {process_1m.data['vector_outputs']['airfare_per_rpk'].loc[2050]:.8f}"
)

## 3. Diagnosing an optimisation run

Every evaluation is kept in the optimisation database, and `run_sweep` writes it to HDF
next to the outputs — so a run can be dissected afterwards without re-optimising it. At
tens of minutes a run, that is the only sane way to work.

Point `RUN` at any HDF written by `01_optimisation_runs.ipynb`.

Note the granularity: rows are *evaluations*, not iterations. With 10 design variables
and finite differences that is roughly eleven rows per gradient.

In [ ]:
from gemseo.algos.optimization_problem import OptimizationProblem

RUN = "results/opt_main_2_6.hdf"

problem = OptimizationProblem.from_hdf(RUN)
print(f"{len(problem.database)} evaluations")
print("recorded functions:", problem.database.get_function_names())

In [ ]:
# Database.get_history_array() cannot stack this problem's history: the objective is
# stored 0-dimensional while the constraints are 5-vectors, and line-search entries hold
# only the objective -- its internal hstack then sees mixed ranks and raises "all the
# input arrays must have same number of dimensions". Build the frame directly. This also
# gives one named column per constraint component rather than opaque indices.
x_names = problem.design_space.get_indexed_variable_names()

rows = []
for x in problem.database.get_x_vect_history():
    row = {}
    for name, value in problem.database[x].items():
        # ravel, not atleast_1d: some values are stored 2-D as well as 0-D.
        values = np.asarray(value, dtype=float).ravel()
        if values.size == 1:
            row[name] = float(values[0])
        else:
            row.update({f"{name}[{i}]": float(v) for i, v in enumerate(values)})
    row.update(dict(zip(x_names, np.asarray(x, dtype=float))))
    rows.append(row)

# Evaluations where the optimiser only needed the objective leave the constraints NaN.
history = pd.DataFrame(rows)

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)
print(f"{len(history)} evaluations x {history.shape[1]} columns")
history.tail(10)

### Did it ever reach the feasible set?

Usually the fastest read on a failed run. If the worst violation never crosses zero the
optimiser never found the feasible set at all; if it crosses and then leaves again,
feasibility was traded for objective.

In [ ]:
constraint_names = [c.name for c in problem.constraints]

worst = np.array(
    [
        max(
            [
                float(np.max(problem.database[x][n]))
                for n in constraint_names
                if n in problem.database[x]
            ]
            or [np.nan]
        )
        for x in problem.database.get_x_vect_history()
    ]
)

# Line-search points carry no constraint values and stay NaN, so read the summary off
# the evaluations that actually have them.
evaluated = worst[~np.isnan(worst)]
reached = np.where(worst <= 1e-4)[0]
print(f"worst violation: first {evaluated[0]:+.4f}  ->  last {evaluated[-1]:+.4f}")
print(f"minimum reached: {evaluated.min():+.4f}")
print(f"constraints evaluated at {evaluated.size} of {worst.size} points")
print("first feasible evaluation:", int(reached[0]) if reached.size else "never")

In [ ]:
from gemseo import execute_post
from gemseo.settings.post import ConstraintsHistory_Settings

# A panel per constraint with the feasible region shaded: a constraint that never comes
# down is visible at a glance. 26 panels need the room -- the 11x11 default packs them
# to a few pixels each.
execute_post(
    problem,
    ConstraintsHistory_Settings(
        constraint_names=constraint_names, save=False, show=True, fig_size=(16.0, 20.0)
    ),
)

# ObjConstrHist would show the same trade-off but formats the worst violation on a log
# scale and raises "cannot convert float NaN to integer" as soon as one evaluation lacks
# a constraint value -- which is every line-search point. Plot it directly instead.
objective_column = next(c for c in history.columns if "surplus" in c)

fig, ax_obj = plt.subplots(figsize=(8, 4))
ax_obj.plot(history.index, history[objective_column], color="tab:blue")
ax_obj.set_xlabel("evaluation")
ax_obj.set_ylabel("objective", color="tab:blue")
ax_obj.tick_params(axis="y", labelcolor="tab:blue")

ax_viol = ax_obj.twinx()
ax_viol.plot(history.index, worst, color="tab:red")
ax_viol.axhline(0.0, color="tab:red", linestyle=":", linewidth=1)
ax_viol.set_ylabel("worst constraint violation", color="tab:red")
ax_viol.tick_params(axis="y", labelcolor="tab:red")

ax_obj.set_title("Objective against feasibility over the run")
plt.tight_layout()

### The same history, in the shape of the problem

`ConstraintsHistory` plots the 26 components against the evaluation index, which is the
right view when a component misbehaves and you need to find it. It is the wrong view for
asking *when* the scenario is under pressure: the five components of each constraint are
five reference years, and the design variables that moved them are five more.

Below, the same numbers indexed by year — this is the per-run version of section 3 of
`02_results.ipynb`, which does it across the budget ladder.


In [ ]:
# The same history in the shape the problem actually has: one panel per constraint,
# each of the five components plotted against the year it is enforced at, and the
# mandate that moved them. Colour is the iterate, black the optimum.
GREEN, BLUE, RED = "#7e9b59", "#092054", "#cb3629"
FLOOR = -1.05  # a pathway using none of its allowed ramp sits at exactly -1

history = R.read_constraint_history(RUN)
cmap = plt.get_cmap("viridis")
shade = [
    "black" if k == len(history) - 1 else cmap(k / max(len(history) - 1, 1))
    for k in range(len(history))
]
weight = [2.4 if k == len(history) - 1 else 1.0 for k in range(len(history))]

fig, axes = plt.subplots(2, 5, figsize=(19, 7.6), gridspec_kw={"hspace": 0.38, "wspace": 0.22})
top, bottom = axes[0], axes[1]

for ax, (name, pretty) in zip(top, R.CONSTRAINT_LABELS.items()):
    for k, entry in enumerate(history):
        ax.plot(
            entry["constraints"].index,
            np.clip(entry["constraints"][name], FLOOR + 0.02, None),
            "-o",
            ms=3,
            color=shade[k],
            lw=weight[k],
            alpha=1 if weight[k] > 2 else 0.7,
            zorder=3 if weight[k] > 2 else 2,
        )
    ax.axhline(0, color=RED, lw=1.2)
    ax.axhspan(FLOOR, 0, color="#f4f4f4", zorder=0)
    ax.set_title(pretty.replace("\n", " "), fontsize=10)
    ax.set_ylim(FLOOR, 0.25)
    if ax is not top[0]:
        ax.set_yticklabels([])
top[0].set_ylabel("Constraint value (clipped at -1)")

# The carbon budget is one number per iterate, so it runs against the iterate.
carbon = [entry["carbon"] for entry in history]
bottom[0].plot(range(len(carbon)), carbon, "-o", ms=3, color="black", lw=1.6)
bottom[0].axhline(0, color=RED, lw=1.2)
bottom[0].set_title("Carbon budget", fontsize=10)
bottom[0].set_xlabel("Iterate")
bottom[0].set_ylabel("Constraint value")

for ax, pathway, colour in [(bottom[1], "biofuel", GREEN), (bottom[2], "electrofuel", BLUE)]:
    for k, entry in enumerate(history):
        ax.plot(
            entry["mandate"].index,
            entry["mandate"][pathway],
            "-o",
            ms=3,
            color=shade[k],
            lw=weight[k],
            alpha=1 if weight[k] > 2 else 0.7,
        )
    ax.set_title(f"{pathway.capitalize()} mandate", fontsize=10, color=colour)
    ax.set_ylim(-2, 102)
bottom[1].set_ylabel("Blending mandate (%)")
bottom[2].set_yticklabels([])
for ax in bottom[3:]:
    ax.axis("off")

for ax in list(top) + list(bottom[:3]):
    ax.grid(alpha=0.3)
    if ax is not bottom[0]:
        ax.set_xticks(R.OPTIM_YEARS)
        ax.set_xlabel("Reference year")

scale = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, len(history) - 1))
fig.colorbar(scale, ax=axes.ravel().tolist(), shrink=0.6, pad=0.01).set_label("iterate")
fig.suptitle(f"{RUN} — {len(history)} iterates, black = optimum", y=0.97)

### When a run ends infeasible

GEMSEO always populates `constraint_values` on the result, but only *prints* them when
there are fewer than 20 — with 26 constraints the listing is silently dropped and you
are left with the verdict alone. Read them off the object instead.

`ineq_tolerance` defaults to **1e-4**, so `is_feasible` can be False on a violation too
small to matter. Sort by magnitude before concluding anything.

In [ ]:
solution = problem.optimum
violated = sorted(
    ((float(np.max(v)), k) for k, v in (solution.constraints or {}).items()), reverse=True
)
print(f"feasible: {solution.is_feasible}   (ineq_tolerance = 1e-4)")
for value, name in violated:
    print(f"  {name:45s} {value:+.6f}  {'VIOLATED' if value > 1e-4 else 'ok'}")

## 4. Migration notes — the traps

**Market parameters defined in `markets.yaml` silently win over the input JSON.** Four
of the paper's assumptions were being overridden with no error raised. Each is now set
in `markets_rte.yaml`:

| parameter | paper | default that was winning | effect if missed |
|---|---|---|---|
| CAGR | 2.2 % | 3.0 % | RPK 2050 +23.6 % |
| partitioning shares | EU (AeroSCOPE) | global | wrong traffic mix |
| efficiency gain | 1.35 %/yr | 2.0 %/yr | cumulative CO₂ −7 % |
| load factor 2050 | 89 % | 85 % | **surplus loss flips sign** |

The last one is the dangerous one: with it wrong the objective read +59.5 Bn€ instead
of −10.4 Bn€, and nothing warned. Anything market-scoped belongs in the markets file,
not in `inputs.json` — which is also why `optimisation_runs.build_process` pushes the
pessimistic efficiency roadmap through `process.parameters` rather than a second
inputs file, as the published notebook did.

**Other things worth knowing**

* `_share_2019` keys are now `_share_last_historical_year`. This one *does* raise.
* A pathway series written `years: []` will not span the historic range, and
  `scenario_cost.py` reads the kerosene emission factor at `prospection_start_year − 1`.
  Give every pathway an explicit span, e.g. `years: [2000, 2050]`.
* GEMSEO's `AutoPyDiscipline` requires each `compute` to `return <name>`, never an
  expression, and the returned names must match the constraint names registered on the
  scenario.
* `setup_mda()` uses `tolerance=1e-10, max_mda_iter=200` on main against `1e-7` on
  `optim_backwards`, so a bare MDA timing comparison between the two branches is not
  like-for-like. The MDO path uses `1e-4` on both and *is* comparable.
* `kerosene_selectivity` is set to 1.0 on both pathways, mapping the old model's
  efficiencies as jet-specific. Worth re-checking against the #158 selectivity fix.

### The MDA residual floor on `main`

`MDAGaussSeidel has reached its maximum number of unsuccessful iterations, but the
normalized residual norm 1.96e-05 is still above the tolerance 1e-10`

Not a failed solve: every discipline output is bit-identical between the last two
sweeps, so the couplings have reached a genuine fixed point. The cause is
`RPKElasticity` clipping the airfare inside `compute()` — the iterate says *x*, the
model computes with *clip(x)*, so the residual compares two different things and can
never reach zero. `demand.model: cagr_elasticity` is what puts that model in the chain.

**PR #157** takes the NaN sentinel out of the coupling vector and the clip out of the
model, declaring the bound through `MDAChain.set_bounds`. Measured on this case: 21
iterations (capped) → 11, residual 1.96e-05 → 3.26e-11, same answers. Results on `main`
are trustworthy, but the MDA does about twice the work it needs to and every
finite-difference gradient pays that twice over. **Run the optimisations on top of #157
if you can.**

### Cross-check before trusting an optimum

The MDA outputs are validated above, but the *constraint* values are not. At the
ReFuelEU point the migrated `aviation_carbon_budget_constraint` evaluates to +0.51, and
the arithmetic relating it to `aviation_carbon_budget` (20.78) and `gross_carbon_budget`
(1135) does not obviously reduce to `(cumulative − budget) / budget`. Before reading
anything into an optimised mandate, evaluate the same design point on both branches and
confirm the six constraints agree in scale — a constraint scaled differently moves the
optimum without moving any MDA output.